# EntoKey v0.6 — BIOSCAN Diptera 30k (selective)

Этот notebook готовит **ровно 30 000 Diptera** из официального BIOSCAN-5M до начала foundation training. Он скачивает полные metadata, но **не скачивает полные image ZIP**: из архивов читаются только выбранные JPEG через HTTP Range.

Прогресс и изображения сохраняются на Google Drive. Если Colab остановится, снова запусти те же ячейки сверху вниз — целые JPEG будут пропущены.

In [1]:
# 1. Подключаем постоянное хранилище
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Берём актуальный EntoKey из GitHub и ставим зависимости
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/SaniyaSani/EntoKey.git'
PROJECT = Path('/content/EntoKey')
if (PROJECT / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-foundation.txt'], check=True)
print('EntoKey ready:', PROJECT)

EntoKey ready: /content/EntoKey


In [3]:
# 3. Все большие файлы остаются на Drive
STORE = Path('/content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan')
STORE.mkdir(parents=True, exist_ok=True)
print('BIOSCAN store:', STORE)
print('Free Drive space (GB):', round(__import__('shutil').disk_usage(STORE).free / 1024**3, 1))

BIOSCAN store: /content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan
Free Drive space (GB): 44.3


## Этап A — metadata и неизменяемый список 30k

Эта ячейка скачивает только официальный metadata archive, проверяет checksum и делает два потоковых прохода по таблице. JPEG пока не скачиваются. Обычно это занимает заметно меньше времени и места, чем images.

In [4]:
# 4. Выбрать ровно 30 000 Diptera по metadata
command = [
    sys.executable, 'scripts/run_bioscan_30k.py',
    '--root', str(STORE),
    '--max-records', '30000',
    '--max-per-taxon', '500',
    '--selection-only',
]
subprocess.run(command, cwd=PROJECT, check=True)
print('SELECTION READY')

SELECTION READY


In [5]:
# 5. Проверить, что выборка действительно равна 30k и увидеть coverage
import json, pandas as pd
report_path = STORE / 'diptera_30k_selection_report.json'
report = json.loads(report_path.read_text())
display(pd.DataFrame([report['selected_by_split']]).T.rename(columns={0: 'images'}))
print('selected:', report['selected'])
print('taxonomic coverage:', report['taxonomic_coverage'])
assert report['selected'] == 30000, report
print('✓ BIOSCAN Diptera selection is exactly 30,000')

,images
key_unseen,750
other_heldout,750
pretrain,12000
test,1500
test_unseen,750
train,12000
val,1500
val_unseen,750


selected: 30000
taxonomic coverage: {'families': 120, 'genera': 1660, 'species': 4465}
✓ BIOSCAN Diptera selection is exactly 30,000


## Этап B — только выбранные JPEG

Эта ячейка может работать долго: она индексирует большие удалённые ZIP, но сам ZIP целиком не сохраняет. Строка `full_archives_downloaded: false` в отчёте подтверждает selective mode. Остановку можно пережить: перезапусти ячейку, и валидные существующие JPEG будут пропущены.

In [ ]:
# 6. Скачать только 30 000 выбранных JPEG и собрать нормализованный Parquet
command = [
    sys.executable, 'scripts/run_bioscan_30k.py',
    '--root', str(STORE),
    '--max-records', '30000',
    '--max-per-taxon', '500',
]
subprocess.run(command, cwd=PROJECT, check=True)
print('BIOSCAN 30k DOWNLOAD READY')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import pandas as pd

STORE = Path(
    '/content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan'
)

print('STORE restored:', STORE)
print('Folder exists:', STORE.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STORE restored: /content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan
Folder exists: True


In [4]:
for path in sorted(STORE.glob('*')):
    print(path.name)


BIOSCAN_5M_Insect_Dataset_metadata_MultiTypes.zip
bioscan5m
diptera_30k_images
diptera_30k_progress.json
diptera_30k_selection.csv
diptera_30k_selection_report.json


In [5]:
import json

progress = json.loads(
    (STORE / 'diptera_30k_progress.json').read_text()
)

print('Готово:', progress['complete'], '/ 30000')
print('Скачано в этой сессии:', progress['downloaded_now'])
print('Последний архив:', progress['active_archive'])
print('Последние ошибки:', progress['errors'][-3:])

Готово: 7994 / 30000
Скачано в этой сессии: 5500
Последний архив: train
Последние ошибки: []


In [6]:
from pathlib import Path
import random
import zipfile

BIOSCAN_IMAGES = Path(
    "/content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan/diptera_30k_images"
)

ANATOMY_DIR = Path(
    "/content/drive/MyDrive/EntoKey/AnatomyPilot"
)
ANATOMY_DIR.mkdir(parents=True, exist_ok=True)

extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
images = [
    path for path in BIOSCAN_IMAGES.rglob("*")
    if path.is_file() and path.suffix.lower() in extensions
]

print("Найдено изображений:", len(images))
assert len(images) >= 20, "В папке пока меньше 20 изображений"

selected = random.Random(42).sample(images, 20)
archive = ANATOMY_DIR / "anatomy_calibration_20.zip"

with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output:
    for image in selected:
        relative_path = image.relative_to(BIOSCAN_IMAGES)
        output.write(image, arcname=str(relative_path))

print("Готово:", archive)
print("В архиве:", len(selected), "изображений")

Найдено изображений: 2494
Готово: /content/drive/MyDrive/EntoKey/AnatomyPilot/anatomy_calibration_20.zip
В архиве: 20 изображений


In [4]:
# Создать ZIP со 100 новыми BIOSCAN-изображениями для Anatomy Batch 02

from google.colab import drive, files
from pathlib import Path
import csv
import json
import random
import shutil

drive.mount("/content/drive")

SOURCE = Path(
    "/content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan/diptera_30k_images"
)

BATCH_DIR = Path("/content/TaxonLens_AnatomyBatch_02_100")
ZIP_PATH = Path("/content/TaxonLens_AnatomyBatch_02_100.zip")

# Первые 20 изображений не выбираем повторно
ALREADY_USED = {
    "BMPHO490-23",
    "BMPHO6997-23",
    "CIBIP15388-23",
    "CIOSA2339-23",
    "CIOSB14309-23",
    "CIOSC2872-23",
    "CISSA33131-23",
    "CRCCF24247-23",
    "GMBZZ2882-23",
    "GMLED7881-23",
    "GMMVA13630-22",
    "GMOFL372-21",
    "GMPPO19598-22",
    "GMSPB32804-23",
    "GMSVC18727-23",
    "HELAA4801-20",
    "HELAB1226-20",
    "HELAC17664-21",
    "HELAC7961-21",
    "PGCBG12528-20",
}

if not SOURCE.exists():
    raise FileNotFoundError(
        f"Папка не найдена:\n{SOURCE}\n"
        "Проверь путь к diptera_30k_images."
    )

extensions = {".jpg", ".jpeg", ".png", ".webp"}

candidates = sorted(
    path
    for path in SOURCE.rglob("*")
    if path.is_file()
    and path.suffix.lower() in extensions
    and path.stem not in ALREADY_USED
)

print("Доступно новых изображений:", len(candidates))

if len(candidates) < 100:
    raise RuntimeError(
        f"Найдено только {len(candidates)} изображений — требуется минимум 100."
    )

# Фиксированный seed: при повторном запуске получится та же выборка
random_generator = random.Random(20260903)
selected = random_generator.sample(candidates, 100)

# Пересоздаём только временную папку Colab
if BATCH_DIR.exists():
    shutil.rmtree(BATCH_DIR)

images_dir = BATCH_DIR / "images"
images_dir.mkdir(parents=True)

manifest_rows = []

for number, source_path in enumerate(selected, start=1):
    destination_name = source_path.name
    destination = images_dir / destination_name

    # Защита на случай одинаковых имён в разных подпапках
    if destination.exists():
        destination_name = f"{number:03d}_{source_path.name}"
        destination = images_dir / destination_name

    shutil.copy2(source_path, destination)

    manifest_rows.append({
        "batch_number": number,
        "file_name": destination_name,
        "specimen_id": source_path.stem,
        "original_relative_path": str(source_path.relative_to(SOURCE)),
    })

with (BATCH_DIR / "selection_manifest.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "batch_number",
            "file_name",
            "specimen_id",
            "original_relative_path",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

batch_info = {
    "batch": "TaxonLens Anatomy Batch 02",
    "images": 100,
    "random_seed": 20260903,
    "excluded_previous_images": len(ALREADY_USED),
    "source_folder": str(SOURCE),
}

(BATCH_DIR / "batch_info.json").write_text(
    json.dumps(batch_info, indent=2),
    encoding="utf-8",
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

created_zip = shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=BATCH_DIR,
)

print("Готово!")
print("Изображений:", len(selected))
print("ZIP:", created_zip)
print("Размер:", round(ZIP_PATH.stat().st_size / 1024 / 1024, 2), "MB")

files.download(str(ZIP_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Доступно новых изображений: 8145
Готово!
Изображений: 100
ZIP: /content/TaxonLens_AnatomyBatch_02_100.zip
Размер: 0.72 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path('/content/EntoKey')
REPO_URL = 'https://github.com/SaniyaSani/EntoKey.git'

if (PROJECT / '.git').exists():
    subprocess.run(
        ['git', '-C', str(PROJECT), 'pull', '--ff-only'],
        check=True
    )
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)],
        check=True
    )

os.chdir(PROJECT)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', 'requirements-foundation.txt'],
    check=True
)

STORE = Path(
    '/content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan'
)

subprocess.run([
    sys.executable,
    'scripts/run_bioscan_30k.py',
    '--root', str(STORE),
    '--skip-metadata-download',
], cwd=PROJECT, check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 7. Финальная строгая проверка перед foundation merge/training
manifest_path = STORE / 'bioscan_diptera_30k_manifest.parquet'
download_report = json.loads((STORE / 'diptera_30k_download_report.json').read_text())
manifest = pd.read_parquet(manifest_path)
assert download_report['complete'] == 30000, download_report
assert download_report['full_archives_downloaded'] is False
assert len(manifest) == 30000
assert set(manifest['order'].dropna().str.casefold()) == {'diptera'}
assert manifest['local_path'].map(lambda p: Path(p).is_file()).all()
foundation_manifest = STORE.parents[1] / 'manifests' / 'bioscan_raw.parquet'
foundation_manifest.parent.mkdir(parents=True, exist_ok=True)
__import__('shutil').copy2(manifest_path, foundation_manifest)
display(manifest.groupby(['source', 'source_split']).size().rename('images').to_frame())
print('✓ READY FOR MASTER CORPUS:', foundation_manifest)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/EntoKey/Foundation_v06/raw/bioscan/diptera_30k_download_report.json'

## Если сервер вернул range-ошибку

В ячейке 6 добавь в `command` строку `'--no-suffix-range'` и запусти снова. Уже загруженные JPEG сохранятся.

После зелёной проверки выше BIOSCAN-часть готова. Следующий этап — собрать такие же manifests iNaturalist, GBIF и DiSSCo, объединить их без дубликатов и только затем запускать DINOv2 embedding shards.